# LeetCode 1232 — Check If It Is a Straight Line bằng Python

**Link đề:** https://leetcode.com/problems/check-if-it-is-a-straight-line/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Geometry

```python
class Solution:
    def checkStraightLine(self, coordinates: list[list[int]]) -> bool:
        ...
```

## Mục tiêu học
1. Ôn lại hình học toạ độ cơ bản: **hệ số góc (slope)**, đường thẳng qua 2 điểm, **tích chéo (cross product)**.
2. Thấy rõ vì sao so sánh slope bằng **phép chia** dễ lỗi (chia cho 0, sai số số thực) và cách **đổi chia thành nhân** để chỉ dùng số nguyên.
3. Nắm unpacking `x, y = p`, `all(...)`, `fractions.Fraction`, `math.gcd`.
4. Đi từ slope-chia (có lỗi) → slope-phân số → **tích chéo** (lời giải chuẩn, `O(n)`, `O(1)`).
5. Trace, chứng minh, test (đường dọc, đường ngang, sai số float), bài tập TODO + đáp án cuối.

## 1. Đề bài và ví dụ

Cho mảng `coordinates`, trong đó `coordinates[i] = [x, y]` là toạ độ một điểm. Kiểm tra các điểm có **cùng nằm trên một đường thẳng** trong mặt phẳng XY không.

| `coordinates` | Kết quả |
|---|:---:|
| `[[1,2],[2,3],[3,4],[4,5],[5,6],[6,7]]` | `True` (đường `y = x + 1`) |
| `[[1,1],[2,2],[3,4],[4,5],[5,6],[7,7]]` | `False` (`[3,4]` lệch khỏi đường `y = x`) |

**Constraints:** `2 <= coordinates.length <= 1000`; `coordinates[i].length == 2`; `-10^4 <= x, y <= 10^4`; **không có điểm trùng**.

Nhận xét ngay: **2 điểm phân biệt luôn thẳng hàng** (qua 2 điểm luôn có đúng 1 đường thẳng).

## 2. Nền tảng toán + Python

### 2.1. Unpacking toạ độ

Mỗi điểm là list 2 phần tử; `x, y = p` tách thành hai biến.

In [1]:
coordinates = [[1, 2], [2, 3], [3, 4]]
for p in coordinates:
    x, y = p
    print("điểm", p, "-> x =", x, ", y =", y)
(x0, y0), (x1, y1) = coordinates[0], coordinates[1]     # unpacking lồng nhau
print("x0, y0, x1, y1 =", x0, y0, x1, y1)

điểm [1, 2] -> x = 1 , y = 2
điểm [2, 3] -> x = 2 , y = 3
điểm [3, 4] -> x = 3 , y = 4
x0, y0, x1, y1 = 1 2 2 3


### 2.2. Hệ số góc (slope)

Đường thẳng qua `A(x0, y0)` và `B(x1, y1)` có hệ số góc

```text
slope = (y1 - y0) / (x1 - x0)      (khi x1 != x0)
```

Ba điểm `A, B, C` thẳng hàng ⇔ `slope(A, B) == slope(A, C)`. Nếu `x1 == x0` thì đường **thẳng đứng** — slope không xác định (chia cho 0).

In [2]:
def slope(p, q):
    return (q[1] - p[1]) / (q[0] - p[0])

print("slope([1,2],[2,3]) =", slope([1, 2], [2, 3]))
print("slope([1,2],[5,6]) =", slope([1, 2], [5, 6]))
try:
    slope([1, 1], [1, 5])
except ZeroDivisionError as e:
    print("Đường thẳng đứng -> ZeroDivisionError:", e)

slope([1,2],[2,3]) = 1.0
slope([1,2],[5,6]) = 1.0
Đường thẳng đứng -> ZeroDivisionError: division by zero


### 2.3. Số thực (float) không chính xác tuyệt đối

Máy tính lưu float theo hệ nhị phân, nên nhiều phép chia cho kết quả xấp xỉ. Hai slope "bằng nhau về toán học" có thể khác nhau ở chữ số cuối.

In [3]:
print(0.1 + 0.2, "== 0.3 ?", 0.1 + 0.2 == 0.3)
print("1/3 =", 1 / 3, "| 3 * (1/3) =", 3 * (1 / 3))
a, b2 = 1 / 49 * 49, 1.0
print("1/49*49 =", a, "== 1.0 ?", a == b2)

0.30000000000000004 == 0.3 ? False
1/3 = 0.3333333333333333 | 3 * (1/3) = 1.0
1/49*49 = 0.9999999999999999 == 1.0 ? False


### 2.4. Đổi phép chia thành phép nhân

So sánh hai phân số `dy1/dx1 == dy2/dx2` (với `dx1, dx2 ≠ 0`) tương đương

```text
dy1 * dx2 == dy2 * dx1
```

(nhân chéo hai vế). Dạng nhân **chỉ dùng số nguyên** → chính xác tuyệt đối, và **không bao giờ chia cho 0**. Đây gọi là **tích chéo (cross product)** của hai vector `(dx1, dy1)` và `(dx2, dy2)`:

```text
cross = dx1 * dy2 - dy1 * dx2      ; thẳng hàng ⇔ cross == 0
```

In [4]:
def cross(o, a, b_):
    # tích chéo của vector OA và OB
    return (a[0] - o[0]) * (b_[1] - o[1]) - (a[1] - o[1]) * (b_[0] - o[0])

print("A,B,C thẳng hàng :", cross([1, 2], [2, 3], [5, 6]))
print("A,B,C không thẳng:", cross([1, 1], [2, 2], [3, 4]))
print("đường dọc        :", cross([1, 1], [1, 5], [1, -3]))

A,B,C thẳng hàng : 0
A,B,C không thẳng: 1
đường dọc        : 0


Ý nghĩa hình học: `|cross|` = **2 lần diện tích tam giác** `OAB`. Ba điểm thẳng hàng ⇔ tam giác "dẹt" ⇔ diện tích 0. Dấu của `cross` cho biết `B` nằm bên trái hay bên phải hướng `O→A`.

### 2.5. `all(...)`

`all(iterable)` trả `True` nếu **mọi** phần tử đúng (dừng sớm ở phần tử sai đầu tiên).

In [5]:
print(all(x > 0 for x in [1, 2, 3]), all(x > 0 for x in [1, -2, 3]), all([]))

True False True


## 3. Cách 1 — Trực giác: so sánh slope bằng phép chia

Lấy slope của 2 điểm đầu làm chuẩn, kiểm tra slope từ điểm đầu tới mọi điểm còn lại. Cần xử lý riêng đường thẳng đứng.

In [6]:
def straight_slope_float(coordinates: list[list[int]]) -> bool:
    (x0, y0), (x1, y1) = coordinates[0], coordinates[1]
    if x1 == x0:                                     # đường thẳng đứng
        return all(x == x0 for x, _ in coordinates)
    base = (y1 - y0) / (x1 - x0)
    for x, y in coordinates[2:]:
        if x == x0:                                  # điểm này nằm trên đường dọc qua A, nhưng A-B không dọc
            return False
        if (y - y0) / (x - x0) != base:
            return False
    return True

print(straight_slope_float([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]))
print(straight_slope_float([[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]]))
print(straight_slope_float([[0, 0], [0, 1], [0, -1]]))

True
False
True


`for x, _ in coordinates` — dấu `_` là tên biến quy ước cho "giá trị không dùng tới".

**Vấn đề của cách này:**
1. Nhiều nhánh đặc biệt (đường dọc) — dễ quên.
2. So sánh `float` bằng `!=` — có thể sai vì làm tròn. Với miền `|x|, |y| ≤ 10^4` thực tế hiếm khi gây lỗi, nhưng đây là thói quen nguy hiểm (xem phản ví dụ ở mục 11).

## 4. Nút thắt và insight

Nút thắt nằm ở **phép chia**. Hai cách bỏ nó:

1. **Giữ slope dạng phân số chính xác**: `Fraction(dy, dx)` hoặc chuẩn hoá `(dy/g, dx/g)` với `g = gcd`. Vẫn phải lo `dx = 0`.
2. **Nhân chéo**: `dy1 * dx2 == dy2 * dx1` — không chia, không float, đường dọc cũng tự động đúng.

Chọn điểm chuẩn `A = coordinates[0]`, `B = coordinates[1]` (khác nhau vì không có điểm trùng). Điểm `P` nằm trên đường `AB` ⇔ `cross(A, B, P) == 0`.

In [7]:
A, B = [1, 1], [2, 2]
for P in [[3, 3], [3, 4], [-5, -5]]:
    c = cross(A, B, P)
    print(f"P={P}: cross={c} -> {'trên đường AB' if c == 0 else 'lệch'}")

P=[3, 3]: cross=0 -> trên đường AB
P=[3, 4]: cross=1 -> lệch
P=[-5, -5]: cross=0 -> trên đường AB


## 5. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — vector chuẩn:** `dx = x1 - x0`, `dy = y1 - y0` (hướng của đường `AB`).

**Đoạn 2 — mỗi điểm `P`:** vector `AP = (x - x0, y - y0)`.

**Đoạn 3 — kiểm tra song song:** `dx * (y - y0) - dy * (x - x0) == 0`. Hai vector song song (cùng phương) và cùng gốc `A` ⇒ `P` nằm trên đường `AB`.

In [8]:
def straight_cross(coordinates: list[list[int]]) -> bool:
    (x0, y0), (x1, y1) = coordinates[0], coordinates[1]
    dx, dy = x1 - x0, y1 - y0                         # đoạn 1
    for x, y in coordinates[2:]:
        if dx * (y - y0) - dy * (x - x0) != 0:        # đoạn 2 + 3
            return False
    return True

for coords in [[[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]],
               [[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]],
               [[0, 0], [0, 1], [0, -1]],
               [[2, 4], [2, 5]]]:
    print(coords, "->", straight_cross(coords))

[[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]] -> True
[[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]] -> False
[[0, 0], [0, 1], [0, -1]] -> True
[[2, 4], [2, 5]] -> True


Viết gọn bằng `all`:

In [9]:
def straight_cross_all(coordinates: list[list[int]]) -> bool:
    (x0, y0), (x1, y1) = coordinates[:2]
    return all((x1 - x0) * (y - y0) == (y1 - y0) * (x - x0) for x, y in coordinates)

print(straight_cross_all([[1, 2], [2, 3], [3, 4]]), straight_cross_all([[1, 1], [2, 2], [3, 4]]))

True False


(Duyệt cả `coordinates` thay vì `coordinates[2:]` cũng được: với `A` và `B` biểu thức tự bằng 0.)

### Cách 3 — Slope dạng phân số chính xác (`Fraction`)

`fractions.Fraction(a, b)` biểu diễn phân số **chính xác** và tự rút gọn. So sánh hai `Fraction` là so sánh chính xác. Vẫn phải xử lý `dx = 0` riêng — ta dùng giá trị đặc biệt `"dọc"`.

In [10]:
from fractions import Fraction

print(Fraction(2, 4), Fraction(-3, 6), Fraction(1, 3) * 3 == 1)

def straight_fraction(coordinates: list[list[int]]) -> bool:
    x0, y0 = coordinates[0]
    def key(p):
        dx, dy = p[0] - x0, p[1] - y0
        return "dọc" if dx == 0 else Fraction(dy, dx)
    slopes = {key(p) for p in coordinates[1:]}        # set comprehension
    return len(slopes) == 1

print(straight_fraction([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]))
print(straight_fraction([[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]]))
print(straight_fraction([[0, 0], [0, 1], [0, -1]]))

1/2 -1/2 True
True
False
True


### Cách 4 — Slope chuẩn hoá bằng `gcd` (không cần `Fraction`)

Rút gọn `(dy, dx)` bằng `g = gcd(dy, dx)` và **chuẩn hoá dấu** (ví dụ luôn để `dx > 0`, hoặc `dx == 0 và dy > 0`). Hai hướng giống nhau ⇔ tuple giống nhau.

In [11]:
from math import gcd

def direction(dx: int, dy: int) -> tuple[int, int]:
    g = gcd(dx, dy)                     # gcd luôn >= 0
    dx, dy = dx // g, dy // g
    if dx < 0 or (dx == 0 and dy < 0):  # chuẩn hoá dấu
        dx, dy = -dx, -dy
    return dx, dy

print(direction(2, 4), direction(-2, -4), direction(0, -7), direction(3, 0))

def straight_gcd(coordinates: list[list[int]]) -> bool:
    x0, y0 = coordinates[0]
    dirs = {direction(x - x0, y - y0) for x, y in coordinates[1:]}
    return len(dirs) == 1

print(straight_gcd([[1, 1], [2, 2], [3, 4]]), straight_gcd([[0, 0], [0, 1], [0, -1]]))

(1, 2) (1, 2) (0, 1) (1, 0)
False True


Chuẩn hoá dấu cần thiết vì điểm có thể nằm **hai phía** của `A`: hướng `(1, 1)` và `(-1, -1)` là cùng một đường.

## 6. TRACE từng bước (tích chéo)

Ví dụ 2: `A = [1,1]`, `B = [2,2]` ⇒ `dx = 1`, `dy = 1`.

| P | x−x0 | y−y0 | dx·(y−y0) | dy·(x−x0) | cross | Kết luận |
|---|:---:|:---:|:---:|:---:|:---:|---|
| [3,4] | 2 | 3 | 3 | 2 | 1 | ≠ 0 → **False** (dừng) |

Ví dụ 1: `A = [1,2]`, `B = [2,3]` ⇒ `dx = 1`, `dy = 1`; mọi điểm `[k, k+1]` có `x−x0 = y−y0` ⇒ cross = 0.

In [12]:
def trace_straight(coordinates: list[list[int]]) -> bool:
    (x0, y0), (x1, y1) = coordinates[0], coordinates[1]
    dx, dy = x1 - x0, y1 - y0
    print(f"A=({x0},{y0}) B=({x1},{y1}) -> dx={dx}, dy={dy}")
    for x, y in coordinates[2:]:
        left, right = dx * (y - y0), dy * (x - x0)
        c = left - right
        print(f"  P=({x},{y}): dx*(y-y0)={left}, dy*(x-x0)={right}, cross={c} -> {'OK' if c == 0 else 'LỆCH -> False'}")
        if c != 0:
            return False
    print("  mọi điểm đều thẳng hàng -> True")
    return True

assert trace_straight([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]) is True
print()
assert trace_straight([[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]]) is False

A=(1,2) B=(2,3) -> dx=1, dy=1
  P=(3,4): dx*(y-y0)=2, dy*(x-x0)=2, cross=0 -> OK
  P=(4,5): dx*(y-y0)=3, dy*(x-x0)=3, cross=0 -> OK
  P=(5,6): dx*(y-y0)=4, dy*(x-x0)=4, cross=0 -> OK
  P=(6,7): dx*(y-y0)=5, dy*(x-x0)=5, cross=0 -> OK
  mọi điểm đều thẳng hàng -> True

A=(1,1) B=(2,2) -> dx=1, dy=1
  P=(3,4): dx*(y-y0)=3, dy*(x-x0)=2, cross=1 -> LỆCH -> False


Trace một đường **thẳng đứng** và một đường **nằm ngang** — cách nhân chéo không cần nhánh riêng:

In [13]:
assert trace_straight([[3, -2], [3, 5], [3, 0], [3, 100]]) is True
print()
assert trace_straight([[-4, 7], [0, 7], [9, 7], [9, 8]]) is False

A=(3,-2) B=(3,5) -> dx=0, dy=7
  P=(3,0): dx*(y-y0)=0, dy*(x-x0)=0, cross=0 -> OK
  P=(3,100): dx*(y-y0)=0, dy*(x-x0)=0, cross=0 -> OK
  mọi điểm đều thẳng hàng -> True

A=(-4,7) B=(0,7) -> dx=4, dy=0
  P=(9,7): dx*(y-y0)=0, dy*(x-x0)=0, cross=0 -> OK
  P=(9,8): dx*(y-y0)=4, dy*(x-x0)=0, cross=4 -> LỆCH -> False


## 7. Code nộp LeetCode

### 7.1. Bản chính — tích chéo

In [14]:
from typing import List

class Solution:
    def checkStraightLine(self, coordinates: List[List[int]]) -> bool:
        (x0, y0), (x1, y1) = coordinates[0], coordinates[1]
        dx, dy = x1 - x0, y1 - y0
        for x, y in coordinates[2:]:
            if dx * (y - y0) != dy * (x - x0):
                return False
        return True

sol = Solution()
print(sol.checkStraightLine([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]))
print(sol.checkStraightLine([[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]]))

True
False


### 7.2. Bản `Fraction`

In [15]:
class SolutionFraction:
    def checkStraightLine(self, coordinates: List[List[int]]) -> bool:
        x0, y0 = coordinates[0]
        slopes = set()
        for x, y in coordinates[1:]:
            slopes.add("dọc" if x == x0 else Fraction(y - y0, x - x0))
        return len(slopes) == 1

assert SolutionFraction().checkStraightLine([[0, 0], [0, 1], [0, -1]]) is True
print("SolutionFraction OK")

SolutionFraction OK


### 7.3. Bản `gcd` chuẩn hoá hướng

In [16]:
class SolutionGcd:
    def checkStraightLine(self, coordinates: List[List[int]]) -> bool:
        x0, y0 = coordinates[0]
        dirs = set()
        for x, y in coordinates[1:]:
            dx, dy = x - x0, y - y0
            g = gcd(dx, dy)
            dx, dy = dx // g, dy // g
            if dx < 0 or (dx == 0 and dy < 0):
                dx, dy = -dx, -dy
            dirs.add((dx, dy))
        return len(dirs) == 1

assert SolutionGcd().checkStraightLine([[1, 1], [2, 2], [3, 4]]) is False
print("SolutionGcd OK")

SolutionGcd OK


## 8. Vì sao đúng?

Gọi `A = (x0, y0)`, `B = (x1, y1)`, `u = B − A = (dx, dy) ≠ (0, 0)` (vì `A ≠ B`).

- Đường thẳng qua `A`, `B` là tập điểm `A + t·u` với `t` thực.
- Điểm `P` nằm trên đường ⇔ vector `v = P − A` **cùng phương** với `u` ⇔ tồn tại `t` với `v = t·u` ⇔ `u.x·v.y − u.y·v.x = 0` (định thức 2×2 bằng 0).
  - Chiều ⇒: `v = t·u` ⇒ `dx·(t·dy) − dy·(t·dx) = 0`.
  - Chiều ⇐: nếu `dx ≠ 0` đặt `t = v.x / dx` ⇒ `v.y = dy·v.x/dx = t·dy`; nếu `dx = 0` thì `dy ≠ 0`, từ `−dy·v.x = 0` ⇒ `v.x = 0`, đặt `t = v.y/dy`.
- Mọi điểm thẳng hàng ⇔ mọi điểm nằm trên đường `AB` ⇔ cross = 0 với mọi `P`. ∎

**Không tràn số / không sai số:** toạ độ ≤ `10^4` ⇒ hiệu ≤ `2·10^4` ⇒ tích ≤ `4·10^8`; Python dùng số nguyên chính xác (ngay cả ở C++/Java vẫn vừa `int` 32-bit).

## 9. Độ phức tạp và so sánh

Gọi `n = len(coordinates)`.

| Cách | Time | Space | Ưu / nhược |
|---|---|---|---|
| Slope bằng phép chia (float) | `O(n)` | `O(1)` | dễ nghĩ; phải xử lý đường dọc; rủi ro sai số float |
| `Fraction` | `O(n log C)` (rút gọn gcd) | `O(n)` set (có thể dừng sớm để `O(1)`) | chính xác; chậm hơn |
| `gcd` + chuẩn hoá | `O(n log C)` | `O(n)` | chính xác; phải chuẩn hoá dấu |
| **Tích chéo** | `O(n)` | `O(1)` | chính xác, không nhánh đặc biệt — chuẩn |

`C` là độ lớn toạ độ (`log C` là chi phí thuật toán Euclid tính gcd).

## 10. Bộ test

Mẫu đề, 2 điểm, đường dọc, đường ngang, điểm nằm hai phía `A`, slope âm, toạ độ biên, và đối chiếu mọi cách trên dữ liệu ngẫu nhiên (sinh điểm **trên** một đường rồi đôi khi xê dịch 1 điểm).

In [17]:
impls = {
    "float": straight_slope_float,
    "cross": straight_cross,
    "cross_all": straight_cross_all,
    "fraction": straight_fraction,
    "gcd": straight_gcd,
    "Solution": Solution().checkStraightLine,
    "SolFraction": SolutionFraction().checkStraightLine,
    "SolGcd": SolutionGcd().checkStraightLine,
}
cases = [
    ([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]], True),
    ([[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]], False),
    ([[0, 0], [5, -3]], True),                              # 2 điểm
    ([[0, 0], [0, 1], [0, -1]], True),                      # đường dọc
    ([[0, 0], [0, 1], [1, 1]], False),                      # A-B dọc, P lệch
    ([[0, 0], [1, 1], [0, 5]], False),                      # P cùng x với A nhưng A-B không dọc
    ([[-4, 7], [0, 7], [9, 7]], True),                      # đường ngang
    ([[2, 2], [4, 4], [-6, -6], [0, 0]], True),             # điểm hai phía A
    ([[0, 0], [2, -1], [-4, 2], [6, -3]], True),            # slope âm
    ([[-10000, -10000], [10000, 10000], [0, 1]], False),    # biên
]
for coords, expected in cases:
    for name, f in impls.items():
        assert f(coords) == expected, (name, coords)
    print(f"{str(coords)[:50]:52} -> {expected} PASS")

[[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]     -> True PASS
[[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]]     -> False PASS
[[0, 0], [5, -3]]                                    -> True PASS
[[0, 0], [0, 1], [0, -1]]                            -> True PASS
[[0, 0], [0, 1], [1, 1]]                             -> False PASS
[[0, 0], [1, 1], [0, 5]]                             -> False PASS
[[-4, 7], [0, 7], [9, 7]]                            -> True PASS
[[2, 2], [4, 4], [-6, -6], [0, 0]]                   -> True PASS
[[0, 0], [2, -1], [-4, 2], [6, -3]]                  -> True PASS
[[-10000, -10000], [10000, 10000], [0, 1]]           -> False PASS


In [18]:
import random

rng = random.Random(1232)
for _ in range(500):
    a = (rng.randint(-20, 20), rng.randint(-20, 20))
    while True:
        u = (rng.randint(-3, 3), rng.randint(-3, 3))
        if u != (0, 0):
            break
    ts = rng.sample(range(-15, 16), rng.randint(2, 8))       # các t khác nhau -> điểm khác nhau
    pts = [[a[0] + t * u[0], a[1] + t * u[1]] for t in ts]
    if len(pts) >= 3 and rng.random() < 0.5:
        i = rng.randrange(len(pts))
        pts[i] = [pts[i][0], pts[i][1] + 1]                  # xê dịch 1 điểm
    if len({tuple(p) for p in pts}) < len(pts):
        continue                                              # tránh điểm trùng (đề không cho phép)
    res = {name: f(pts) for name, f in impls.items()}
    assert len(set(res.values())) == 1, (pts, res)
print("500 bộ điểm ngẫu nhiên: mọi cách khớp nhau")

500 bộ điểm ngẫu nhiên: mọi cách khớp nhau


## 11. Lỗi người mới hay gặp

### Lỗi 1: Chia cho 0 khi gặp đường thẳng đứng

In [19]:
def wrong_divide_by_zero(coordinates):
    (x0, y0), (x1, y1) = coordinates[:2]
    base = (y1 - y0) / (x1 - x0)
    return all((y - y0) / (x - x0) == base for x, y in coordinates[1:])

try:
    wrong_divide_by_zero([[0, 0], [0, 1], [0, -1]])
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e, "-> dùng nhân chéo thay vì chia")

ZeroDivisionError: division by zero -> dùng nhân chéo thay vì chia


### Lỗi 2: So sánh float bằng `==` / `!=` (sai số làm tròn)

Trong miền toạ độ của đề, các slope `dy/dx` hiếm khi "va" nhau do sai số, nhưng với toạ độ lớn hơn thì có thể. Ví dụ hai phân số **khác nhau** nhưng cho cùng một float:

In [20]:
p, q = 10**16 + 1, 10**16
print("Toán học: (10^16 + 1)/10^16 != 1  ->", Fraction(p, q) != 1)
print("Float   : (10^16 + 1)/10^16 == 1.0 ->", p / q == 1.0)

pts = [[0, 0], [10**16, 10**16], [10**16, 10**16 + 1]]     # không thẳng hàng
print("float slope:", straight_slope_float(pts), "| cross:", straight_cross(pts))

Toán học: (10^16 + 1)/10^16 != 1  -> True
Float   : (10^16 + 1)/10^16 == 1.0 -> True
float slope: True | cross: False


Cách float kết luận **sai** là thẳng hàng; tích chéo (số nguyên) cho đáp án đúng. Bài học: **tránh chia khi chỉ cần so sánh tỉ lệ**.

### Lỗi 3: Dùng slope giữa các điểm **kề nhau** nhưng quên đường dọc / hoặc chỉ so 3 điểm đầu

Chỉ kiểm tra 3 điểm đầu là chưa đủ — phải kiểm tra **mọi** điểm.

### Lỗi 4: Dùng `gcd` nhưng quên chuẩn hoá dấu

In [21]:
def wrong_gcd_no_sign(coordinates):
    x0, y0 = coordinates[0]
    dirs = set()
    for x, y in coordinates[1:]:
        dx, dy = x - x0, y - y0
        g = gcd(dx, dy)
        dirs.add((dx // g, dy // g))
    return len(dirs) == 1

print("sai :", wrong_gcd_no_sign([[2, 2], [4, 4], [-6, -6]]), "| đúng:", straight_cross([[2, 2], [4, 4], [-6, -6]]))

sai : False | đúng: True


`(1, 1)` và `(-1, -1)` là cùng một đường nhưng khác tuple → sai.

### Lỗi 5: Viết `if n == 2: return True` rồi đọc `coordinates[2]` không kiểm tra

Với cách tích chéo, vòng `for` trên `coordinates[2:]` rỗng khi `n == 2` → tự trả `True`; không cần nhánh riêng.

## 12. Mẹo nhận dạng pattern

- **"Thẳng hàng", "cùng hướng", "song song", "cùng slope"** → tích chéo `dx1*dy2 - dy1*dx2 == 0`.
- **Muốn so sánh hai tỉ lệ** `a/b` và `c/d` với số nguyên → nhân chéo `a*d == b*c` (nhớ cẩn thận dấu khi so `<`/`>`).
- **Muốn gom các điểm theo hướng/slope** (đếm, dùng làm key của dict) → chuẩn hoá `(dx/g, dy/g)` + dấu, hoặc `Fraction`.

Bài cùng họ: 149 Max Points on a Line (Hard — gom slope bằng gcd), 1037 Valid Boomerang (3 điểm không thẳng hàng ⇔ cross ≠ 0), 593 Valid Square, 812 Largest Triangle Area (công thức diện tích = |cross|/2).

## 13. Biến thể

### 13.1. Valid Boomerang (LeetCode 1037)

3 điểm là boomerang nếu phân biệt và **không** thẳng hàng ⇔ `cross ≠ 0` (cross = 0 cũng bao gồm trường hợp có điểm trùng).

In [22]:
def is_boomerang(points: list[list[int]]) -> bool:
    return cross(points[0], points[1], points[2]) != 0

print(is_boomerang([[1, 1], [2, 3], [3, 2]]), is_boomerang([[1, 1], [2, 2], [3, 3]]), is_boomerang([[0, 0], [0, 0], [1, 1]]))

True False False


### 13.2. Diện tích tam giác lớn nhất (LeetCode 812)

Diện tích = `|cross| / 2`. Thử mọi bộ 3 điểm (`n ≤ 50`).

In [23]:
from itertools import combinations

def largest_triangle_area(points: list[list[int]]) -> float:
    return max(abs(cross(a, b_, c)) / 2 for a, b_, c in combinations(points, 3))

print(largest_triangle_area([[0, 0], [0, 1], [1, 0], [0, 2], [2, 0]]))   # 2.0

2.0


### 13.3. Max Points on a Line (LeetCode 149 — Hard)

Với mỗi điểm làm gốc, gom các điểm khác theo **hướng chuẩn hoá** (gcd + dấu); nhóm lớn nhất + 1 là số điểm trên một đường qua gốc đó.

In [24]:
from collections import Counter

def max_points(points: list[list[int]]) -> int:
    if len(points) <= 2:
        return len(points)
    best = 0
    for i, (x0, y0) in enumerate(points):
        cnt = Counter(direction(x - x0, y - y0) for j, (x, y) in enumerate(points) if j != i)
        best = max(best, max(cnt.values()) + 1)
    return best

print(max_points([[1, 1], [2, 2], [3, 3]]), max_points([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]))   # 3 4

3 4


## 14. Bài tập (TODO — không gọi khi chạy)

**Bài 1 (tính tay).** `A=[2,1]`, `B=[4,5]`, `P=[5,7]`, `Q=[0,-4]`. Tính `cross(A,B,P)` và `cross(A,B,Q)`. Bộ `[A,B,P,Q]` có thẳng hàng không?

**Bài 2 (dự đoán output).** `gcd(-6, 4)`, `Fraction(6, -4)`, `0.1 * 3 == 0.3`.

**Bài 3 (điền TODO).**

In [25]:
def exercise_straight(coordinates: list[list[int]]) -> bool:
    (x0, y0), (x1, y1) = coordinates[0], coordinates[1]
    # TODO 1: tính dx, dy
    # TODO 2: với mỗi điểm còn lại, nếu dx*(y-y0) != dy*(x-x0) thì trả False
    # TODO 3: trả True
    pass

# assert exercise_straight([[1, 2], [2, 3], [3, 4]]) is True
# assert exercise_straight([[0, 0], [0, 1], [1, 1]]) is False

**Bài 4 (viết lại).** Viết `exercise_cross(o, a, b)` và dùng nó để giải bài (không viết lại công thức trong vòng lặp).

In [26]:
def exercise_cross(o, a, b_):
    # TODO
    pass

def exercise_straight_v2(coordinates):
    # TODO: dùng exercise_cross(coordinates[0], coordinates[1], p)
    pass

# assert exercise_straight_v2([[3, -2], [3, 5], [3, 0]]) is True

**Bài 5 (biến thể).** Cho danh sách điểm, trả về **chỉ số điểm đầu tiên** (tính từ 2) làm hỏng tính thẳng hàng; nếu tất cả thẳng hàng trả `-1`.

In [27]:
def exercise_first_bad(coordinates: list[list[int]]) -> int:
    # TODO
    pass

# assert exercise_first_bad([[1, 1], [2, 2], [3, 4], [4, 5]]) == 2
# assert exercise_first_bad([[1, 2], [2, 3], [3, 4]]) == -1

**Bài 6 (mini challenge).** Cho `points` (có thể > 1000 điểm, không trùng), kiểm tra tồn tại một đường thẳng chứa **tất cả trừ tối đa 1 điểm**. Gợi ý: trong 3 điểm đầu, ít nhất hai điểm thuộc đường cần tìm.

In [28]:
def exercise_all_but_one(points: list[list[int]]) -> bool:
    # TODO
    pass

# assert exercise_all_but_one([[0, 0], [1, 1], [2, 2], [5, 0]]) is True
# assert exercise_all_but_one([[0, 0], [1, 1], [2, 0], [5, 7]]) is False
# assert exercise_all_but_one([[9, 9], [0, 0], [1, 1], [2, 2]]) is True

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Vector AB = (2, 4) | vector AP, AQ | `2*(y-1) - 4*(x-2)` |
| 2 | gcd luôn không âm | Fraction tự rút gọn và chuyển dấu lên tử | float không chính xác |
| 3 | Vector chuẩn từ 2 điểm đầu | `for x, y in coordinates[2:]` | so sánh hai tích |
| 4 | Hàm nhỏ tái sử dụng | `(a.x-o.x)*(b.y-o.y) - (a.y-o.y)*(b.x-o.x)` | `all(exercise_cross(A, B, p) == 0 for p in ...)` |
| 5 | Như bài gốc nhưng trả chỉ số | `enumerate(coordinates[2:], start=2)` | trả `i` khi cross ≠ 0 |
| 6 | Thử 3 đường: (p0,p1), (p0,p2), (p1,p2) | với mỗi đường đếm điểm lệch | đúng nếu có đường mà số điểm lệch ≤ 1 |

## 16. Đáp án tham khảo

**Bài 1:** `AB = (2, 4)`. `AP = (3, 6)` → `2·6 − 4·3 = 0`. `AQ = (−2, −5)` → `2·(−5) − 4·(−2) = −2`. Không thẳng hàng (Q lệch).

**Bài 2:** `2`, `Fraction(-3, 2)`, `False`.

In [29]:
def answer_straight(coordinates):
    (x0, y0), (x1, y1) = coordinates[0], coordinates[1]
    dx, dy = x1 - x0, y1 - y0
    for x, y in coordinates[2:]:
        if dx * (y - y0) != dy * (x - x0):
            return False
    return True

def answer_cross(o, a, b_):
    return (a[0] - o[0]) * (b_[1] - o[1]) - (a[1] - o[1]) * (b_[0] - o[0])

def answer_straight_v2(coordinates):
    A, B = coordinates[0], coordinates[1]
    return all(answer_cross(A, B, p) == 0 for p in coordinates[2:])

def answer_first_bad(coordinates):
    A, B = coordinates[0], coordinates[1]
    for i, p in enumerate(coordinates[2:], start=2):
        if answer_cross(A, B, p) != 0:
            return i
    return -1

def answer_all_but_one(points):
    if len(points) <= 3:
        return True
    for i, j in ((0, 1), (0, 2), (1, 2)):
        A, B = points[i], points[j]
        bad = sum(1 for p in points if answer_cross(A, B, p) != 0)
        if bad <= 1:
            return True
    return False

assert answer_cross([2, 1], [4, 5], [5, 7]) == 0 and answer_cross([2, 1], [4, 5], [0, -4]) == -2
assert (gcd(-6, 4), Fraction(6, -4), 0.1 * 3 == 0.3) == (2, Fraction(-3, 2), False)
assert answer_straight([[1, 2], [2, 3], [3, 4]]) is True and answer_straight([[0, 0], [0, 1], [1, 1]]) is False
assert answer_straight_v2([[3, -2], [3, 5], [3, 0]]) is True
assert answer_first_bad([[1, 1], [2, 2], [3, 4], [4, 5]]) == 2
assert answer_first_bad([[1, 2], [2, 3], [3, 4]]) == -1
assert answer_all_but_one([[0, 0], [1, 1], [2, 2], [5, 0]]) is True
assert answer_all_but_one([[0, 0], [1, 1], [2, 0], [5, 7]]) is False
assert answer_all_but_one([[9, 9], [0, 0], [1, 1], [2, 2]]) is True
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
(x0, y0), (x1, y1) = coordinates[0], coordinates[1]
dx, dy = x1 - x0, y1 - y0
for x, y in coordinates[2:]:
    if dx * (y - y0) != dy * (x - x0):   # nhân chéo, không chia
        return False
return True
```

**Câu thần chú:** *"So tỉ lệ thì nhân chéo, đừng chia."*

### Checklist trước khi submit
- [ ] Không có phép chia (tránh `ZeroDivisionError` với đường dọc)?
- [ ] Không so sánh float bằng `==`?
- [ ] Kiểm tra **mọi** điểm, không chỉ vài điểm đầu?
- [ ] `n == 2` tự trả `True`?
- [ ] Tên method `checkStraightLine(self, coordinates)`, trả `bool`?